# Temporal Alignment — News and Liquidity Data using OHLCV

This notebook aligns the cleaned GDELT news dataset with the hourly WTI price and 
liquidity data from Yahoo Finance. Each news article is matched to the closest 
trading hour using timestamp rounding. Articles published outside trading hours 
(nights, weekends) are discarded as no liquidity data is available for those 
periods. The resulting dataset merges article metadata with contemporaneous 
liquidity metrics (log volume, price range, Amihud ratio), and constitutes the 
central modeling dataset for the thesis. It is saved to 
`01_data/processed/gdelt_wti_aligned.csv`.

### General imports

In [1]:
import pandas as pd
import numpy as np

### Load hourly price data

In [2]:
df_price = pd.read_csv("../01_data/raw/price/wti_hourly_raw.csv", 
                       index_col='Datetime', 
                       parse_dates=True)
df_price.index = pd.to_datetime(df_price.index, utc=True)

print(f"Price records: {len(df_price)}")
print(f"Range: {df_price.index.min()} to {df_price.index.max()}")

Price records: 11219
Range: 2024-03-11 10:00:00+00:00 to 2026-03-11 10:00:00+00:00


### Load news and align to nearest trading hour

For each article, the publication timestamp is rounded to the nearest hour and matched against the price dataset. Articles published before the earliest available price record are discarded.

In [3]:
df_news = pd.read_csv("../01_data/processed/gdelt_wti_with_body.csv", parse_dates=['datetime'])
df_news['datetime'] = pd.to_datetime(df_news['datetime'], utc=True)

price_start = df_price.index.min()
price_end = df_price.index.max()

# Discard articles outside the price data range
df_news_filtered = df_news[
    (df_news['datetime'] >= price_start) &
    (df_news['datetime'] <= price_end)
].reset_index(drop=True)

df_news_filtered = df_news_filtered.copy()
df_news_filtered['datetime_hour'] = df_news_filtered['datetime'].dt.round('h')

print(f"Articles before filter: {len(df_news)}")
print(f"Articles after filter: {len(df_news_filtered)}")
print(f"Discarded (outside price range): {len(df_news) - len(df_news_filtered)}")

# Prepare price data with liquidity features
df_price_reset = df_price.copy()
df_price_reset.index.name = 'datetime_hour'
df_price_reset = df_price_reset.reset_index()

# Compute liquidity features
df_price_reset['log_volume'] = np.log(df_price_reset['Volume'] + 1)
df_price_reset['price_range'] = df_price_reset['High'] - df_price_reset['Low']
df_price_reset['log_return'] = np.log(df_price_reset['Close'] / df_price_reset['Close'].shift(1))
df_price_reset['amihud'] = df_price_reset['log_return'].abs() / (df_price_reset['Volume'] + 1)

# Merge news with price data on nearest hour
df_aligned = df_news_filtered.merge(
    df_price_reset[['datetime_hour', 'Close', 'Volume', 'log_volume', 'price_range', 'log_return', 'amihud']],
    on='datetime_hour',
    how='left'
)

print(f"Articles aligned: {len(df_aligned)}")
print(f"Without price (outside trading hours): {df_aligned['Volume'].isna().sum()}")
print(df_aligned[['title', 'datetime', 'Close', 'log_volume', 'price_range']].head(5))

Articles before filter: 3290
Articles after filter: 2934
Discarded (outside price range): 356
Articles aligned: 2934
Without price (outside trading hours): 611
                                               title  \
0   Will an Economic Rebound in China Lift Prices ?    
1  Oil sprints higher on Monday with Gaza ceasefi...   
2  Oil up on supply concerns as Russia orders out...   
3  Local Gas Prices Lower than Counties North and...   
4                  Tennessee gas prices jump 9 cents   

                   datetime      Close  log_volume  price_range  
0 2024-03-25 04:15:00+00:00  81.150002    6.415097     0.050003  
1 2024-03-25 15:15:00+00:00  81.970001   10.558803     0.380005  
2 2024-03-25 18:30:00+00:00  82.129997   10.116742     0.470001  
3 2024-03-25 21:00:00+00:00        NaN         NaN          NaN  
4 2024-03-25 21:15:00+00:00        NaN         NaN          NaN  


### Discard articles without price data

Articles published outside NYMEX trading hours (nights and weekends) cannot be matched to a trading hour and are therefore discarded. Since no market activity is observable at those times, including them would not contribute meaningful liquidity measurements to the analysis.

In [4]:
df_final = df_aligned.dropna(subset=['Close', 'Volume']).reset_index(drop=True)

print(f"Articles with price data: {len(df_final)}")
print(f"Temporal range: {df_final['datetime'].min()} to {df_final['datetime'].max()}")
print(f"\nLiquidity statistics:")
print(df_final[['log_volume', 'price_range', 'amihud']].describe().round(4))

Articles with price data: 2323
Temporal range: 2024-03-25 04:15:00+00:00 to 2026-02-20 19:30:00+00:00

Liquidity statistics:
       log_volume  price_range     amihud
count   2323.0000    2323.0000  2323.0000
mean       8.1725       0.3791     0.0001
std        2.4290       0.2761     0.0008
min        0.0000       0.0000     0.0000
25%        7.3476       0.2000     0.0000
50%        8.6618       0.3100     0.0000
75%        9.7374       0.4900     0.0000
max       11.7523       3.4200     0.0191


### Save data

In [5]:
filename = "gdelt_wti_aligned.csv"
df_final.to_csv(f"../01_data/processed/{filename}", index=False)
print(f"Saved to 01_data/processed/{filename}")

Saved to 01_data/processed/gdelt_wti_aligned.csv
